# Live Demo 4 — Optimising the DCGAN generator for deployment

We take the generator trained in `03_dcgan_fashionmnist.ipynb` and try the classic deployment optimisations — with
**honest measurements** of size, CPU latency and output quality:

| Technique | Expected effect on *this* network | Section |
|---|---|---|
| PyTorch dynamic INT8 quantization | **nothing** — no `nn.Linear` layers (trap!) | (a) |
| Conv-BN fusion | same output, fewer ops | (a2) |
| FP16 / BF16 weights | ½ size; CPU speed depends on hardware support | (b) |
| ONNX export + ONNX Runtime | faster runtime, esp. at batch 1 | (c) |
| ONNX Runtime **static** INT8 (QDQ) | ~¼ size, small quality loss, speed depends on kernels | (d) |
| L1 unstructured pruning 30–90 % | same dense size & speed, compresses better, quality degrades | (e) |
| Knowledge distillation → small student | much smaller + faster, some quality loss | (f) |

**Quality metric:** the generator has no labels/accuracy. We measure **fidelity to the FP32 teacher** —
PSNR (dB) between the optimised model's output and the FP32 output *for the same latent z* (higher = closer;
> 40 dB is visually identical, < 20 dB clearly different) — plus **visual inspection**, which is what matters for a GAN.

> **Trainer note:** needs `checkpoints/dcgan_fashionmnist_G.pth` from notebook 03. The only slow cell is the distillation
> (≈ 2–3 min, cached with `USE_CACHED=True`).

## 0. Setup — load the trained generator

In [ ]:
from pathlib import Path
import time, copy, io, gzip, warnings
import numpy as np
import pandas as pd
import torch, torch.nn as nn
import torch.nn.utils.prune as prune
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import onnx, onnxruntime as ort

USE_CACHED = True
CKPT_DIR, OUT_DIR = Path("checkpoints"), Path("outputs")
OUT_DIR.mkdir(exist_ok=True)
G_PATH = CKPT_DIR / "dcgan_fashionmnist_G.pth"
assert G_PATH.exists(), "Run 03_dcgan_fashionmnist.ipynb first (it creates the generator checkpoint)"
torch.manual_seed(0)

class Generator(nn.Module):                       # identical to notebook 03
    def __init__(self, nz=100, ngf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf * 4, 7, 1, 0, bias=False), nn.BatchNorm2d(ngf * 4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False), nn.BatchNorm2d(ngf * 2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, 1, 4, 2, 1, bias=False), nn.Tanh())
    def forward(self, z):
        return self.net(z)

ckpt = torch.load(G_PATH, map_location="cpu", weights_only=True)
NZ = ckpt["nz"]
G = Generator(NZ, ckpt["ngf"]); G.load_state_dict(ckpt["G"]); G.eval()
print(f"torch {torch.__version__} | onnxruntime {ort.__version__} | torch threads {torch.get_num_threads()}")
print(G)

### Measurement helpers
* **Size** = bytes of the serialised weights (`state_dict` / `.onnx` file).
* **Latency** = median of many timed runs after warm-up (first calls include memory allocation & kernel selection).
  Batch 1 = interactive "generate one image"; batch 64 = throughput.
* **PSNR vs FP32** on 256 fixed latent vectors.

In [ ]:
Z_EVAL = torch.randn(256, NZ, 1, 1, generator=torch.Generator().manual_seed(42))
Z1, Z64 = Z_EVAL[:1], Z_EVAL[:64]
with torch.inference_mode():
    REF = G(Z_EVAL)                                  # FP32 reference outputs in [-1, 1]

def state_dict_mb(model):
    buf = io.BytesIO(); torch.save(model.state_dict(), buf); return buf.tell() / 1e6

def gzip_mb(model):
    buf = io.BytesIO(); torch.save(model.state_dict(), buf); return len(gzip.compress(buf.getvalue())) / 1e6

def latency_ms(fn, runs=100, warmup=10, min_warmup_s=1.0):
    # Warm up for at least `warmup` calls AND `min_warmup_s` seconds: an idle CPU needs time to raise its clock and
    # wake the thread pool - otherwise whichever variant is measured first looks slower than it is.
    t_end = time.perf_counter() + min_warmup_s
    n = 0
    while n < warmup or time.perf_counter() < t_end:
        fn(); n += 1
    # 3 trials, keep the best trial's median (like `timeit`): background load can only slow a trial down.
    medians = []
    for _ in range(3):
        ts = []
        for _ in range(runs):
            t0 = time.perf_counter(); fn(); ts.append((time.perf_counter() - t0) * 1e3)
        medians.append(float(np.median(ts)))
    return min(medians)

def torch_latency(model, dtype=torch.float32):
    z1, z64 = Z1.to(dtype), Z64.to(dtype)
    with torch.inference_mode():
        return latency_ms(lambda: model(z1)), latency_ms(lambda: model(z64), runs=30)

def psnr(out, ref=None):
    ref = REF if ref is None else ref
    a, b = (out.float() + 1) / 2, (ref.float() + 1) / 2        # map to [0, 1]
    mse = torch.mean((a - b) ** 2).item()
    return float("inf") if mse == 0 else 10 * np.log10(1.0 / mse)

results, samples = [], {}
def record(name, size_mb, lat1, lat64, out, note=""):
    results.append({"variant": name, "size (MB)": round(size_mb, 3), "latency bs1 (ms)": round(lat1, 3),
                    "latency bs64 (ms)": round(lat64, 2), "PSNR vs FP32 (dB)": round(psnr(out), 1), "note": note})
    samples[name] = out[:12].float()
    print(results[-1])

n_params = sum(p.numel() for p in G.parameters())
l1, l64 = torch_latency(G)
record("FP32 eager (baseline)", state_dict_mb(G), l1, l64, REF, f"{n_params/1e6:.2f} M params")

## (a) PyTorch dynamic INT8 quantization — why it does *nothing* here

`torch.ao.quantization.quantize_dynamic` only swaps **`nn.Linear` / `nn.LSTM` / `nn.GRU`** layers (weights stored
as INT8, activations quantised on-the-fly). It is great for Transformers/LSTMs (Day 2), but our generator is
**100 % `ConvTranspose2d`**, so nothing gets quantised. Let's prove it rather than believe it:

> Note: in PyTorch 2.9 `torch.ao.quantization` prints a *deprecation* warning — it is scheduled for removal in 2.10;
> the successor is the separate `torchao` library. For conv nets, the practical path today is **ONNX Runtime static
> quantization** (section d) or `torchao` PT2E quantization.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)
    G_dq = torch.ao.quantization.quantize_dynamic(copy.deepcopy(G), {nn.Linear}, dtype=torch.qint8)
changed = [n for (n, m1), (_, m2) in zip(G.named_modules(), G_dq.named_modules()) if type(m1) is not type(m2)]
print("modules replaced by dynamic quantization:", changed if changed else "NONE")
print(f"size before {state_dict_mb(G):.3f} MB -> after {state_dict_mb(G_dq):.3f} MB")
print("layer types in G:", sorted({type(m).__name__ for m in G.modules() if len(list(m.children())) == 0}))

## (a2) Conv-BN fusion (free lunch at inference time)
In `eval()` mode a BatchNorm is just a fixed per-channel affine transform `y = γ·(x−μ)/σ + β`, so it can be folded
into the preceding (transposed) convolution's weights and bias. Same maths, fewer ops and memory passes.
We use the fused model for the ONNX export below.

In [ ]:
from torch.nn.utils.fusion import fuse_conv_bn_eval

def fuse_generator(g):
    g = copy.deepcopy(g).eval()
    L = list(g.net)
    fused = [fuse_conv_bn_eval(L[0], L[1], transpose=True), L[2],
             fuse_conv_bn_eval(L[3], L[4], transpose=True), L[5], L[6], L[7]]
    g.net = nn.Sequential(*fused)
    return g

G_fused = fuse_generator(G)
with torch.inference_mode():
    out_fused = G_fused(Z_EVAL)
print("max |fused - original| =", (out_fused - REF).abs().max().item())
l1, l64 = torch_latency(G_fused)
record("FP32 eager, Conv-BN fused", state_dict_mb(G_fused), l1, l64, out_fused, "exact same function")

## (b) FP16 / BF16 weights
Halving the bits halves the model size. **Speed on CPU is another story:** most laptop CPUs have no native FP16 matrix
units (FP16 kernels often convert internally), and BF16 is only fast on CPUs with AVX-512-BF16/AMX (recent Xeons).
On GPUs/NPUs FP16 is usually ~2× faster. Measure, don't assume!

In [ ]:
for dtype, name in [(torch.float16, "FP16 eager"), (torch.bfloat16, "BF16 eager")]:
    try:
        g = copy.deepcopy(G).to(dtype)
        with torch.inference_mode():
            out = g(Z_EVAL.to(dtype))
        l1, l64 = torch_latency(g, dtype)
        record(name, state_dict_mb(g), l1, l64, out, "size halves; CPU speed hardware-dependent")
    except Exception as e:
        print(f"{name}: not supported on this CPU/PyTorch build -> {e!r}")

## (c) ONNX export + ONNX Runtime
ONNX is a framework-neutral graph format; ONNX Runtime (ORT) applies graph optimisations and uses highly tuned CPU
kernels, with much lower per-call overhead than eager PyTorch — this matters most at **batch 1**.
We export with a **dynamic batch axis** so one file serves any batch size.

In [ ]:
ONNX_FP32 = CKPT_DIR / "dcgan_G_fp32.onnx"
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    torch.onnx.export(G_fused, (Z1,), str(ONNX_FP32), opset_version=17, dynamo=False,
                      input_names=["z"], output_names=["image"],
                      dynamic_axes={"z": {0: "batch"}, "image": {0: "batch"}})
onnx.checker.check_model(onnx.load(str(ONNX_FP32)))
print("ops:", [n.op_type for n in onnx.load(str(ONNX_FP32)).graph.node])

def ort_session(path):
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])

def ort_eval(path, name, note):
    sess = ort_session(path)
    out = torch.from_numpy(sess.run(None, {"z": Z_EVAL.numpy()})[0])
    z1, z64 = Z1.numpy(), Z64.numpy()
    l1 = latency_ms(lambda: sess.run(None, {"z": z1}))
    l64 = latency_ms(lambda: sess.run(None, {"z": z64}), runs=30)
    record(name, path.stat().st_size / 1e6, l1, l64, out, note)
    return out

out_onnx = ort_eval(ONNX_FP32, "ONNX Runtime FP32", "fused, dynamic batch")
print("max |ORT - PyTorch| =", (out_onnx - REF).abs().max().item())

## (d) ONNX Runtime INT8 quantization

* **Dynamic** quantization (ORT `quantize_dynamic`) — like PyTorch's — only handles MatMul/Gemm-style ops with
  on-the-fly activation scales. Let's check how many `ConvTranspose` nodes it quantises.
* **Static** quantization (`quantize_static`) also fixes the *activation* scales ahead of time from a **calibration
  set**. For a generator the calibration data is trivial and perfect: the input distribution *is* z ~ N(0, 1)!
  We use QDQ format (Quantize → DeQuantize pairs around each op, which ORT fuses into INT8 kernels where available)
  with per-channel weight scales.

In [ ]:
from onnxruntime.quantization import (quantize_dynamic, quantize_static, CalibrationDataReader,
                                      QuantFormat, QuantType)
import logging; logging.getLogger().setLevel(logging.ERROR)       # silence "consider pre-processing" hints

ONNX_DYN  = CKPT_DIR / "dcgan_G_int8_dynamic.onnx"
ONNX_INT8 = CKPT_DIR / "dcgan_G_int8_static.onnx"

quantize_dynamic(str(ONNX_FP32), str(ONNX_DYN), weight_type=QuantType.QInt8)
dyn_ops = [n.op_type for n in onnx.load(str(ONNX_DYN)).graph.node]
print("ORT dynamic quantization ->", "no quantized ops (graph unchanged)" if not any("Quant" in o or "Integer" in o for o in dyn_ops) else dyn_ops)

class RandomLatentReader(CalibrationDataReader):
    # 32 batches x 64 random latent vectors = 2048 calibration samples
    def __init__(self, n_batches=32, bs=64, seed=0):
        g = np.random.default_rng(seed)
        self.data = iter([{"z": g.standard_normal((bs, NZ, 1, 1)).astype(np.float32)} for _ in range(n_batches)])
    def get_next(self):
        return next(self.data, None)

t0 = time.perf_counter()
quantize_static(str(ONNX_FP32), str(ONNX_INT8), RandomLatentReader(), quant_format=QuantFormat.QDQ,
                per_channel=True, activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)
print(f"static quantization + calibration took {time.perf_counter()-t0:.1f}s")
ops = [n.op_type for n in onnx.load(str(ONNX_INT8)).graph.node]
print("quantized graph op counts:", {o: ops.count(o) for o in sorted(set(ops))})
out_int8 = ort_eval(ONNX_INT8, "ONNX Runtime INT8 (static, QDQ)", "calibrated on random z")

## (e) Pruning: 30 / 50 / 70 / 90 % L1-unstructured

`torch.nn.utils.prune.l1_unstructured` zeroes the individual weights with the smallest |w| in every ConvTranspose layer.

**Honest caveat:** the tensors stay dense — the zeros are still stored and multiplied — so **size on disk and
latency do not change**. You only gain if (1) you compress the file (zeros compress well → see gzip column), or
(2) you use a sparse inference engine, or (3) you use **structured** pruning (remove whole channels → genuinely
smaller layers, usually followed by fine-tuning). Unstructured pruning is mostly a research / compression tool.

In [ ]:
prune_rows = []
for amount in [0.3, 0.5, 0.7, 0.9]:
    gp = copy.deepcopy(G)
    for m in gp.modules():
        if isinstance(m, nn.ConvTranspose2d):
            prune.l1_unstructured(m, name="weight", amount=amount)
            prune.remove(m, "weight")                                    # make it permanent
    w = torch.cat([m.weight.flatten() for m in gp.modules() if isinstance(m, nn.ConvTranspose2d)])
    with torch.inference_mode():
        out = gp(Z_EVAL)
    l1, l64 = torch_latency(gp)
    record(f"Pruned {int(amount*100)}% (L1 unstructured)", state_dict_mb(gp), l1, l64, out,
           f"sparsity {(w == 0).float().mean().item():.0%}, gzip {gzip_mb(gp):.2f} MB vs {gzip_mb(G):.2f} MB")

fig, axes = plt.subplots(1, 5, figsize=(20, 4.6))
for ax, name in zip(axes, ["FP32 eager (baseline)"] + [f"Pruned {p}% (L1 unstructured)" for p in (30, 50, 70, 90)]):
    grid = make_grid(samples[name][:9], nrow=3, normalize=True, value_range=(-1, 1), pad_value=1)
    ax.imshow(grid.permute(1, 2, 0)); ax.axis("off")
    ax.set_title(name.replace(" (L1 unstructured)", "") + f"\nPSNR {next(r['PSNR vs FP32 (dB)'] for r in results if r['variant'] == name)} dB")
plt.suptitle("Pruning without fine-tuning: quality degrades as sparsity grows")
plt.tight_layout(); plt.savefig(OUT_DIR / "gan_opt_pruning.png", dpi=90); plt.show()

## (f) Knowledge distillation: train a 5× smaller student generator

The teacher is a function z → image. We can train a **smaller** generator (channel width `ngf=16` instead of 64)
to reproduce the teacher's output for the *same* z, with a simple pixel loss — no real data, no discriminator needed!

`loss = L1(student(z), teacher(z))`, z ~ N(0, 1) freshly sampled every step (an infinite, free training set).

Unlike pruning, the student is **genuinely** smaller and faster (dense, fewer channels).
Trade-off: a pixel loss tends to make the student's samples a bit **blurrier** (it averages what it cannot represent).

In [ ]:
STUDENT_NGF, DISTILL_ITERS, BS = 16, 3000, 128
S_PATH = CKPT_DIR / f"dcgan_student_ngf{STUDENT_NGF}.pth"
student = Generator(NZ, STUDENT_NGF)

if USE_CACHED and S_PATH.exists():
    sck = torch.load(S_PATH, map_location="cpu", weights_only=True)
    student.load_state_dict(sck["G"]); distill_log = sck["log"]
    print(f"Loaded cached student (distilled in {sck['time_s']:.0f}s)")
else:
    torch.manual_seed(1)
    opt = torch.optim.Adam(student.parameters(), lr=2e-3, betas=(0.5, 0.999))
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, DISTILL_ITERS)
    distill_log, t0 = [], time.perf_counter()
    student.train()
    for it in range(1, DISTILL_ITERS + 1):
        z = torch.randn(BS, NZ, 1, 1)
        with torch.no_grad():
            target = G(z)                          # teacher in eval mode
        loss = nn.functional.l1_loss(student(z), target)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        distill_log.append(loss.item())
        if it % 250 == 0:
            print(f"[distill it {it}/{DISTILL_ITERS}] L1={np.mean(distill_log[-250:]):.4f}  "
                  f"elapsed {time.perf_counter()-t0:.0f}s", flush=True)
    t_distill = time.perf_counter() - t0
    student.eval()
    torch.save({"G": student.state_dict(), "log": distill_log, "time_s": t_distill, "ngf": STUDENT_NGF, "nz": NZ}, S_PATH)
    print(f"distillation took {t_distill:.0f}s -> saved {S_PATH}")

student.eval()
with torch.inference_mode():
    out_student = student(Z_EVAL)
l1, l64 = torch_latency(student)
s_params = sum(p.numel() for p in student.parameters())
record(f"Distilled student (ngf={STUDENT_NGF})", state_dict_mb(student), l1, l64, out_student,
       f"{s_params/1e6:.2f} M params ({n_params/s_params:.1f}x fewer)")
plt.figure(figsize=(6, 3)); plt.plot(np.convolve(distill_log, np.ones(50) / 50, mode="valid"))
plt.xlabel("iteration"); plt.ylabel("L1 to teacher"); plt.title("Distillation loss"); plt.tight_layout(); plt.show()

## Final comparison

In [ ]:
df = pd.DataFrame(results).set_index("variant")
base = df.loc["FP32 eager (baseline)"]
df["size vs base"] = (df["size (MB)"] / base["size (MB)"]).round(2).astype(str) + "x"
df["speed-up bs1"] = (base["latency bs1 (ms)"] / df["latency bs1 (ms)"]).round(2).astype(str) + "x"
df["speed-up bs64"] = (base["latency bs64 (ms)"] / df["latency bs64 (ms)"]).round(2).astype(str) + "x"
pd.set_option("display.max_colwidth", 80)
display(df)
df.to_csv(OUT_DIR / "gan_optimization_results.csv")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
names = df.index.tolist(); y = np.arange(len(names))
for ax, col in zip(axes, ["size (MB)", "latency bs1 (ms)", "latency bs64 (ms)"]):
    ax.barh(y, df[col], color=["C3" if n.startswith("FP32 eager (b") else "C0" for n in names])
    ax.set_yticks(y); ax.set_yticklabels(names if col == "size (MB)" else [""] * len(names)); ax.invert_yaxis()
    ax.set_title(col + "  (lower is better)")
    for yi, v in zip(y, df[col]): ax.text(v, yi, f" {v:g}", va="center", fontsize=8)
plt.tight_layout(); plt.savefig(OUT_DIR / "gan_optimization_chart.png", dpi=90); plt.show()

### Side by side: same latent vectors through every variant

In [ ]:
show = ["FP32 eager (baseline)", "FP16 eager", "ONNX Runtime INT8 (static, QDQ)",
        "Pruned 50% (L1 unstructured)", "Pruned 90% (L1 unstructured)", f"Distilled student (ngf={STUDENT_NGF})"]
show = [s for s in show if s in samples]
fig, axes = plt.subplots(len(show), 1, figsize=(13, 1.6 * len(show)))
for ax, name in zip(axes, show):
    grid = make_grid(samples[name][:12], nrow=12, normalize=True, value_range=(-1, 1), pad_value=1)
    ax.imshow(grid.permute(1, 2, 0)); ax.set_xticks([]); ax.set_yticks([])
    ax.set_ylabel(name.replace(" (L1 unstructured)", "").replace(" (static, QDQ)", ""), rotation=0, ha="right", va="center", fontsize=9)
plt.suptitle("fp32 vs fp16 vs int8 vs pruned vs distilled (same z per column)")
plt.tight_layout(); plt.savefig(OUT_DIR / "gan_opt_side_by_side.png", dpi=90, bbox_inches="tight"); plt.show()

## Take-aways for the class

1. **Know which layers a technique touches.** Dynamic quantization is a one-liner — and a no-op for conv/deconv nets.
2. **Free wins first:** eval-mode fusion and a better runtime (ONNX Runtime) change no maths and are often the
   biggest latency win at batch 1.
3. **Lower precision** reliably shrinks the file (FP16 ½, INT8 ¼); whether it is *faster* depends on the hardware's
   kernels — always benchmark on the deployment target.
4. **Unstructured pruning** does not make a dense model smaller or faster by itself.
5. **Distillation** is the only technique here that changes the *architecture* → genuinely smaller and faster, at the
   price of a training run and some quality (sharpness).
6. For GANs, judge quality **visually** (and with FID for real projects) — a PSNR vs the teacher only measures fidelity.